In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# שלום עולם

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
דוגמה זו מכילה שני חלקים. תחילה תיצור תוכנית קוונטית פשוטה ותריץ אותה על יחידת עיבוד קוונטי (QPU). מכיוון שמחקר קוונטי אמיתי דורש תוכניות מאוד מקיפות, בחלק השני ([הרחבה למספרים גדולים של קיוביטים](#scale-to-large-numbers-of-qubits)), תרחיב את התוכנית הפשוטה לרמת שימושיות.

## התקנה ואימות
1. אם עדיין לא התקנת את Qiskit, מצא הוראות במדריך [Quickstart](/guides/quick-start).

    - התקן את Qiskit Runtime כדי להריץ משימות על חומרה קוונטית:

        ```bash
        pip install qiskit-ibm-runtime
        ```

    - הגדר סביבה להרצת מחברות Jupyter באופן מקומי:

        ```bash
        pip install jupyter
        ```

2. הגדר את האימות שלך לגישה לחומרה קוונטית דרך ה-[Open Plan](/guides/plans-overview#open-plan) החינמי.

    (אם קיבלת הזמנה בדואר אלקטרוני להצטרף לחשבון, בצע את [השלבים למשתמשים מוזמנים](/guides/cloud-setup-invited) במקום.)

    - היכנס ל-[IBM Quantum Platform](https://quantum.cloud.ibm.com/) כדי להתחבר או ליצור חשבון.
         > **Note:** אם אתה מתחבר דרך שרת פרוקסי, עליך להשתמש ב-Qiskit Runtime גרסה 0.44.0 ומעלה.
    - צור את מפתח ה-API שלך (הידוע גם כ-*API token*) ב[לוח הבקרה](https://quantum.cloud.ibm.com/), ואז העתק אותו למיקום מאובטח.
    - עבור לדף [Instances](https://quantum.cloud.ibm.com/instances) ומצא את ה-instance שברצונך להשתמש בו. רחף מעל ה-CRN שלו ולחץ להעתקה.

    - שמור את פרטי הגישה שלך באופן מקומי עם הקוד הזה:

        ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        QiskitRuntimeService.save_account(
        token="<your-api-key>", # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
        instance="<CRN>", # Optional
        )
        ```

3. עכשיו תוכל להשתמש בקוד Python הזה בכל פעם שתרצה לאמת מול Qiskit Runtime Service:
    ```python
        from qiskit_ibm_runtime import QiskitRuntimeService

        # Run every time you need the service
        service = QiskitRuntimeService()
    ```
> **Info:** אם אתה משתמש במחשב ציבורי או בסביבה לא מאובטחת אחרת, בצע את [הוראות האימות הידני](/guides/cloud-setup-untrusted) במקום כדי לשמור על אבטחת פרטי הגישה שלך.
## יצירה והרצה של תוכנית קוונטית פשוטה
ארבעת השלבים לכתיבת תוכנית קוונטית באמצעות תבניות Qiskit הם:

1.  מיפוי הבעיה לפורמט קוונטי-מקורי.

2.  אופטימיזציה של ה-Circuit והאופרטורים.

3.  הרצה באמצעות פונקציית primitive קוונטית.

4.  ניתוח התוצאות.

### שלב 1. מיפוי הבעיה לפורמט קוונטי-מקורי
בתוכנית קוונטית, *Circuit קוונטיים* הם הפורמט המקורי לייצוג הוראות קוונטיות, ו*אופרטורים* מייצגים את ה-observables שיש למדוד. בעת יצירת Circuit, בדרך כלל תיצור אובייקט [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) חדש, ואז תוסיף הוראות ברצף.
תא הקוד הבא יוצר Circuit שמייצר *מצב Bell*, שהוא מצב שבו שני Qubit שזורים לחלוטין זה בזה.

> **Note:** ה-Qiskit SDK משתמש בספירת ביטים LSb 0, שבה הספרה ה-$n$ השווה ל-$1 \ll n$ או $2^n$. לפרטים נוספים, ראה את הנושא [סדר ביטים ב-Qiskit SDK](/guides/bit-ordering).

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from matplotlib import pyplot as plt
# Uncomment the next line if you want to use a simulator:
# from qiskit_ibm_runtime.fake_provider import FakeBelemV2


# Create a new circuit with two qubits
qc = QuantumCircuit(2)

# Add a Hadamard gate to qubit 0
qc.h(0)

# Perform a controlled-X gate on qubit 1, controlled by qubit 0
qc.cx(0, 1)

# Return a drawing of the circuit using MatPlotLib ("mpl").
# These guides are written by using Jupyter notebooks, which
# display the output of the last line of each cell.
# If you're running this in a script, use `print(qc.draw())` to
# print a text drawing.
qc.draw("mpl")

<Image src="../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/930ca3b6-0.svg)

ראה [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#quantumcircuit-class) בתיעוד לכל הפעולות הזמינות.
בעת יצירת Circuit קוונטיים, עליך גם לשקול איזה סוג נתונים ברצונך לקבל לאחר הביצוע. Qiskit מספק שתי דרכים להחזרת נתונים: ניתן לקבל התפלגות הסתברות עבור קבוצת Qubit שתבחר למדוד, או לקבל את ערך הציפייה של observable. הכן את העומס שלך למדידת ה-Circuit באחת משתי הדרכים הללו עם [primitives של Qiskit](/guides/get-started-with-primitives) (מוסבר בפירוט ב[שלב 3](#step-3-execute-using-the-quantum-primitives)).

דוגמה זו מודדת ערכי ציפייה באמצעות מודול המשנה `qiskit.quantum_info`, המצוין באמצעות אופרטורים (אובייקטים מתמטיים המשמשים לייצוג פעולה או תהליך שמשנים מצב קוונטי). תא הקוד הבא יוצר שישה אופרטורי Pauli דו-קיוביטיים: `IZ`, `IX`, `ZI`, `XI`, `ZZ`, ו-`XX`.

In [2]:
# Set up six different observables.

observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

> **Note:** כאן, משהו כמו האופרטור `ZZ` הוא קיצור לכפל טנזורי $Z\otimes Z$, שמשמעותו מדידת Z על Qubit 1 ו-Z על Qubit 0 יחד, וקבלת מידע על הקורלציה בין Qubit 1 ל-Qubit 0. ערכי ציפייה כאלה נכתבים בדרך כלל גם כ-$\langle Z_1 Z_0 \rangle$.
> 
> אם המצב שזור, אז מדידת $\langle Z_1 Z_0 \rangle$ צריכה להיות שונה ממדידת $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$. עבור המצב השזור הספציפי שנוצר על ידי ה-Circuit שלנו המתואר לעיל, מדידת $\langle Z_1 Z_0 \rangle$ צריכה להיות 1 ומדידת $\langle I_1 \otimes Z_0 \rangle \langle Z_1 \otimes I_0 \rangle$ צריכה להיות אפס.
<span id="optimize"></span>

### שלב 2. אופטימיזציה של ה-Circuit והאופרטורים

בעת ביצוע Circuit על מכשיר, חשוב לייעל את קבוצת ההוראות שה-Circuit מכיל ולמזער את העומק הכולל (בקירוב מספר ההוראות) של ה-Circuit. הדבר מבטיח שתקבל את התוצאות הטובות ביותר האפשריות על ידי הפחתת השפעות שגיאות ורעש. בנוסף, הוראות ה-Circuit חייבות להתאים ל-[Instruction Set Architecture (ISA)](/guides/transpile#instruction-set-architecture) של Backend המכשיר ולקחת בחשבון את שערי הבסיס וקישוריות ה-Qubit של המכשיר.

הקוד הבא מאתחל מכשיר אמיתי להגשת משימה אליו ומשנה את ה-Circuit והאופרטורים כך שיתאימו ל-ISA של ה-Backend. הוא דורש שכבר [שמרת את פרטי הגישה שלך](/guides/cloud-setup)

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(simulator=False, operational=True)

# Convert to an ISA circuit and layout-mapped observables.
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

isa_circuit.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/9a901271-0.svg)

### שלב 3. הרצה באמצעות ה-primitives הקוונטיים
מחשבים קוונטיים יכולים לייצר תוצאות אקראיות, לכן בדרך כלל אוספים דגימה של הפלטים על ידי הרצת ה-Circuit פעמים רבות. ניתן להעריך את ערך ה-observable באמצעות מחלקת `Estimator`. `Estimator` הוא אחד משני [primitives](/guides/get-started-with-primitives); השני הוא `Sampler`, שניתן להשתמש בו לקבלת נתונים ממחשב קוונטי. לאובייקטים אלה יש שיטת `run()` שמבצעת את בחירת ה-Circuit, האופרטורים והפרמטרים (אם רלוונטי), באמצעות [primitive unified bloc (PUB).](/guides/primitives#sampler)

In [4]:
# Construct the Estimator instance.

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

mapped_observables = [
    observable.apply_layout(isa_circuit.layout) for observable in observables
]

# One pub, with one circuit to run against five different observables.
job = estimator.run([(isa_circuit, mapped_observables)])

# Use the job ID to retrieve your job data later
print(f">>> Job ID: {job.job_id()}")

>>> Job ID: d5k96q4jt3vs73ds5tgg


After a job is submitted, you can wait until either the job is completed within your current python instance, or use the `job_id` to retrieve the data at a later time.  (See the [section on retrieving jobs](/docs/guides/monitor-job#retrieve-job-results-at-a-later-time) for details.)

After the job completes, examine its output through the job's `result()` attribute.

In [5]:
# This is the result of the entire submission.  You submitted one Pub,
# so this contains one inner result (and some metadata of its own).
job_result = job.result()

# This is the result from our single pub, which had six observables,
# so contains information on all six.
pub_result = job.result()[0]

In [6]:
# Check there are six observables.
# If not, edit the comments in the previous cell and update this test.
assert len(pub_result.data.evs) == 6

<Admonition type="note" title="Alternative: run the example using a simulator">
  When you run your quantum program on a real device, your workload must wait in a queue before it runs. To save time, you can instead use the following code to run this small workload on the [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) with the Qiskit Runtime local testing mode. Note that this is only possible for a small circuit. When you scale up in the next section, you will need to use a real device.

  ```python

  # Use the following code instead if you want to run on a simulator:

  from qiskit_ibm_runtime.fake_provider import FakeBelemV2
  backend = FakeBelemV2()
  estimator = Estimator(backend)

  # Convert to an ISA circuit and layout-mapped observables.

  pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
  isa_circuit = pm.run(qc)
  mapped_observables = [
      observable.apply_layout(isa_circuit.layout) for observable in observables
  ]

  job = estimator.run([(isa_circuit, mapped_observables)])
  result = job.result()

  # This is the result of the entire submission.  You submitted one Pub,
  # so this contains one inner result (and some metadata of its own).

  job_result = job.result()

  # This is the result from our single pub, which had five observables,
  # so contains information on all five.

  pub_result = job.result()[0]
  ```
</Admonition>

### Step 4. Analyze the results

The analyze step is typically where you might post-process your results using, for example, measurement error mitigation or zero noise extrapolation (ZNE). You might feed these results into another workflow for further analysis or prepare a plot of the key values and data. In general, this step is specific to your problem.  For this example, plot each of the expectation values that were measured for our circuit.

The expectation values and standard deviations for the observables you specified to Estimator are accessed through the job result's `PubResult.data.evs` and `PubResult.data.stds` attributes. To obtain the results from Sampler, use the `PubResult.data.meas.get_counts()` function, which will return a `dict` of measurements in the form of bitstrings as keys and counts as their corresponding values. For more information, see [Get started with Sampler.](/docs/guides/get-started-with-primitives#get-started-with-sampler)

In [ ]:
# Plot the result

values = pub_result.data.evs

errors = pub_result.data.stds

# plotting graph
plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg" alt="Output of the previous code cell" />

> **Note:** כאשר אתה מריץ את התוכנית הקוונטית שלך על מכשיר אמיתי, העומס שלך חייב להמתין בתור לפני שירוץ. כדי לחסוך זמן, תוכל להשתמש בקוד הבא כדי להריץ עומס קטן זה על [`fake_provider`](../api/qiskit-ibm-runtime/fake-provider) עם מצב הבדיקה המקומי של Qiskit Runtime. שים לב שזה אפשרי רק עבור Circuit קטן. כאשר תרחיב בסעיף הבא, יהיה עליך להשתמש במכשיר אמיתי.
> 
>   ```python
> 
>   # Use the following code instead if you want to run on a simulator:
> 
>   from qiskit_ibm_runtime.fake_provider import FakeBelemV2
>   backend = FakeBelemV2()
>   estimator = Estimator(backend)
> 
>   # Convert to an ISA circuit and layout-mapped observables.
> 
>   pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
>   isa_circuit = pm.run(qc)
>   mapped_observables = [
>       observable.apply_layout(isa_circuit.layout) for observable in observables
>   ]
> 
>   job = estimator.run([(isa_circuit, mapped_observables)])
>   result = job.result()
> 
>   # This is the result of the entire submission.  You submitted one Pub,
>   # so this contains one inner result (and some metadata of its own).
> 
>   job_result = job.result()
> 
>   # This is the result from our single pub, which had five observables,
>   # so contains information on all five.
> 
>   pub_result = job.result()[0]
>   ```
### שלב 4. ניתוח התוצאות
שלב הניתוח הוא בדרך כלל המקום שבו תוכל לעבד את תוצאותיך בדיעבד באמצעות, לדוגמה, הפחתת שגיאות מדידה או אקסטרפולציה של אפס רעש (ZNE). ייתכן שתזין תוצאות אלה לתוך תהליך עבודה אחר לניתוח נוסף או תכין גרף של הערכים והנתונים המרכזיים. באופן כללי, שלב זה ספציפי לבעיה שלך. לדוגמה זו, צייר גרף של כל ערכי הציפייה שנמדדו עבור ה-Circuit שלנו.

ערכי הציפייה והסטיות התקניות עבור ה-observables שציינת ל-Estimator נגישים דרך המאפיינים `PubResult.data.evs` ו-`PubResult.data.stds` של תוצאת המשימה. כדי לקבל את התוצאות מ-Sampler, השתמש בפונקציה `PubResult.data.meas.get_counts()`, שתחזיר `dict` של מדידות בצורת מחרוזות ביטים כמפתחות וספירות כערכים המתאימים. למידע נוסף, ראה [התחל עם Sampler.](/guides/get-started-with-primitives#get-started-with-sampler)

In [8]:
# Make sure the results follow the claim from the previous markdown cell.
# This can happen when the device occasionally behaves strangely. If this cell
# fails, you may just need to run the notebook again.

_results = {obs: val for obs, val in zip(observables_labels, values)}
for _label in ["IZ", "IX", "ZI", "XI"]:
    assert abs(_results[_label]) < 0.2
for _label in ["XX", "ZZ"]:
    assert _results[_label] > 0.8

![Output of the previous code cell](../docs/images/guides/hello-world/extracted-outputs/87143fcc-0.svg)

שים לב שעבור Qubit 0 ו-1, ערכי הציפייה העצמאיים של X ו-Z הם 0, בעוד שהקורלציות (`XX` ו-`ZZ`) הן 1. זהו סימן היכר של שזירה קוונטית.

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """This function will create a qiskit.QuantumCircuit (qc) for an n-qubit GHZ state.

    Args:
        n (int): Number of qubits in the n-qubit GHZ state

    Returns:
        QuantumCircuit: Quantum circuit that generate the n-qubit GHZ state, assuming all qubits start in the 0 state
    """
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n is not a valid input")
    return qc


# Create a new circuit with two qubits (first argument) and two classical
# bits (second argument)
n = 100
qc = get_qc_for_n_qubit_GHZ_state(n)

## הרחבה למספרים גדולים של קיוביטים
בחישוב קוונטי, עבודה בקנה מידה של שימושיות היא קריטית לקידום בתחום. עבודה כזו דורשת ביצוע חישובים בקנה מידה גדול הרבה יותר; עבודה עם Circuit שעשויים להשתמש ביותר מ-100 Qubit וביותר מ-1000 Gate. דוגמה זו מדגימה כיצד תוכל לבצע עבודה בקנה מידה שימושיות על QPU של IBM&reg; על ידי יצירה וניתוח של מצב GHZ של 100 Qubit. היא משתמשת בתהליך העבודה של תבניות Qiskit ומסתיימת במדידת ערך הציפייה $\langle Z_0 Z_i \rangle$ עבור כל Qubit.

### שלב 1. מיפוי הבעיה
כתוב פונקציה שמחזירה `QuantumCircuit` המכין מצב GHZ של $n$ Qubit (בעצם מצב Bell מורחב), ואז השתמש בפונקציה זו להכנת מצב GHZ של 100 Qubit ואסוף את ה-observables שיש למדוד.

In [ ]:
# ZZII...II, ZIZI...II, ... , ZIII...IZ
operator_strings = [
    "Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)
]
print(operator_strings)
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

['ZZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII', 'ZIIIIIIIIIZIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII

לאחר מכן, מפה לאופרטורים המעניינים. דוגמה זו משתמשת באופרטורי `ZZ` בין Qubit כדי לבחון את ההתנהגות ככל שהם מתרחקים זה מזה. ערכי ציפייה שגויים יותר ויותר (פגומים) בין Qubit מרוחקים יחשפו את רמת הרעש הקיימת.

In [ ]:
service = QiskitRuntimeService()

backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Step 3. Execute on hardware

Submit the job and enable error suppression by using a technique to reduce errors called [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) The resilience level specifies how much resilience to build against errors. Higher levels generate more accurate results, at the expense of longer processing times.  For further explanation of the options set in the following code, see [Configure error mitigation for Qiskit Runtime.](/docs/guides/configure-error-mitigation)

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

# Create an Estimator object
estimator = Estimator(backend, options=options)

In [13]:
# Submit the circuit to Estimator
job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

d5k9mmqvcahs73a1ni3g


### שלב 3. הרצה על חומרה
הגש את המשימה והפעל דיכוי שגיאות באמצעות טכניקה להפחתת שגיאות הנקראת [dynamical decoupling.](../api/qiskit-ibm-runtime/options-dynamical-decoupling-options) רמת החוסן מציינת כמה חוסן לבנות מפני שגיאות. רמות גבוהות יותר מייצרות תוצאות מדויקות יותר, על חשבון זמני עיבוד ארוכים יותר. להסבר נוסף על האפשרויות שנקבעו בקוד הבא, ראה [הגדרת הפחתת שגיאות עבור Qiskit Runtime.](/guides/configure-error-mitigation)

In [ ]:
# data
data = list(range(1, len(operators) + 1))  # Distance between the Z operators
result = job.result()[0]
values = result.data.evs  # Expectation value at each Z operator.
values = [
    v / values[0] for v in values
]  # Normalize the expectation values to evaluate how they decay with distance.

# plotting graph
plt.plot(data, values, marker="o", label="100-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()

<Image src="../docs/images/guides/hello-world/extracted-outputs/de91ebd0-0.svg" alt="Output of the previous code cell" />

The previous plot shows that as the distance between qubits increases, the signal decays because of the presence of noise.

## Next steps

<Admonition type="tip" title="Recommendations">
  -   Try one of these tutorials:
      - [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
      - Solve optimization problems using [QAOA](/docs/tutorials/quantum-approximate-optimization-algorithm)
      - Train [quantum kernel](/docs/tutorials/quantum-kernel-training) models for machine learning tasks
  - Find detailed installation instructions in the [Install Qiskit](/docs/guides/install-qiskit) guide.
  - If you prefer not to install Qiskit locally, read about options to use Qiskit in an [online development environment.](/docs/guides/online-lab-environments)
  - To save multiple account credentials or to specify other account options, see detailed instructions in the [Save your login credentials](/docs/guides/save-credentials#save-your-access-credentials) guide.
</Admonition>